# Dim_Date Generation Notebook

This notebook generates a comprehensive date dimension table with:
- Enterprise-grade calendar attributes
- Group fiscal year calculations
- Multi-division fiscal year support (configurable)
- Holiday information

**Prerequisites:**
1. Lakehouse attached to this notebook
2. Configuration file at: `Files/config/dimension_config.json` in the lakehouse
3. Target Fabric Warehouse configured

## 1. Configuration & Setup

In [ ]:
# CONFIGURATION
# Get workspace and lakehouse IDs dynamically from the attached lakehouse
workspace_id = notebookutils.runtime.context.get('workspaceId', '')
lakehouse_id = notebookutils.runtime.context.get('lakehouseId', '')

if not workspace_id or not lakehouse_id:
    raise ValueError(
        "Unable to retrieve workspace or lakehouse ID. "
        "Please ensure a lakehouse is attached to this notebook."
    )

# Configuration file path in lakehouse using IDs
config_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Files/config/dimension_config.json"

print(f"Workspace ID: {workspace_id}")
print(f"Lakehouse ID: {lakehouse_id}")
print(f"Configuration path: {config_path}")

In [ ]:
# Import required libraries
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, lit, when, date_format, dayofmonth, dayofyear, dayofweek,
    weekofyear, month, quarter, year, last_day, concat, lpad,
    row_number, datediff, date_add, to_date, expr, floor, sequence
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType
from datetime import datetime, timedelta
import json

print("Libraries imported successfully")

In [ ]:
# Load configuration from lakehouse Files
try:
    config_df = spark.read.option("multiline", "true").json(config_path)
    config_json = config_df.first().asDict()
    
    # Parse nested JSON structures
    config = {
        "date_range": config_df.select("date_range.*").first().asDict(),
        "group_fiscal_year": config_df.select("group_fiscal_year.*").first().asDict(),
        "divisions": [row.asDict() for row in config_df.select("divisions").first()[0]],
        "holidays": config_df.select("holidays.*").first().asDict()
    }
    
    print("Configuration loaded successfully")
    print(f"Date range: {config['date_range']['start_year']} to {config['date_range']['end_year']}")
    print(f"Number of divisions: {len(config['divisions'])}")
    print(f"Divisions: {[d['code'] for d in config['divisions']]}")
except Exception as e:
    print(f"Error loading configuration: {e}")
    print("Please ensure the config file exists at the specified path")
    raise

## 2. Table Creation (DDL)

In [ ]:
# Build DDL dynamically based on divisions
division_columns = []
for div in config['divisions']:
    div_code = div['code']
    division_columns.extend([
        f"    {div_code}_FiscalYear INT,",
        f"    {div_code}_FiscalQuarter INT,",
        f"    {div_code}_FiscalMonth INT,",
        f"    {div_code}_FiscalYearQuarter VARCHAR(10),",
        f"    {div_code}_FiscalYearMonth VARCHAR(10),",
        f"    {div_code}_FiscalWeekOfYear INT,",
        f"    {div_code}_FiscalDayOfYear INT,"
    ])

division_columns_sql = "\n".join(division_columns)

ddl_sql = f"""
DROP TABLE IF EXISTS Dim_Date;

CREATE TABLE Dim_Date (
    -- Primary Key
    DateKey INT NOT NULL,
    Date DATE NOT NULL,
    
    -- Calendar Attributes
    Year INT,
    Quarter INT,
    Month INT,
    Day INT,
    YearMonth INT,
    YearQuarter VARCHAR(10),
    MonthName VARCHAR(20),
    MonthNameShort VARCHAR(3),
    DayName VARCHAR(20),
    DayNameShort VARCHAR(3),
    DayOfWeek INT,
    DayOfMonth INT,
    DayOfYear INT,
    DayOfQuarter INT,
    WeekOfYear INT,
    WeekOfMonth INT,
    ISOWeekOfYear INT,
    CalendarQuarter VARCHAR(10),
    CalendarSemester INT,
    
    -- Date Flags
    IsWeekend BIT,
    IsWeekday BIT,
    IsLastDayOfMonth BIT,
    IsLastDayOfQuarter BIT,
    IsLastDayOfYear BIT,
    IsLeapYear BIT,
    
    -- Holiday Information
    IsHoliday BIT,
    HolidayName VARCHAR(100),
    
    -- Group Fiscal Year
    Group_FiscalYear INT,
    Group_FiscalQuarter INT,
    Group_FiscalMonth INT,
    Group_FiscalYearQuarter VARCHAR(10),
    Group_FiscalYearMonth VARCHAR(10),
    Group_FiscalWeekOfYear INT,
    Group_FiscalDayOfYear INT,
    
    -- Division Fiscal Years
{division_columns_sql}
    
    -- Metadata
    LoadDate TIMESTAMP
);
"""

print("Executing DDL...")
for statement in ddl_sql.split(';'):
    if statement.strip():
        spark.sql(statement)

print("Table Dim_Date created successfully")

## 3. Data Generation

In [ ]:
# Generate date range
start_year = config['date_range']['start_year']
end_year = config['date_range']['end_year']

start_date = f"{start_year}-01-01"
end_date = f"{end_year}-12-31"

# Create date sequence
dates_df = spark.sql(f"""
    SELECT sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day) as date_array
""").select(expr("explode(date_array)").alias("Date"))

print(f"Generated {dates_df.count()} dates from {start_date} to {end_date}")
dates_df.show(5)

In [ ]:
# Calculate calendar attributes
dim_date_df = dates_df \
    .withColumn("DateKey", date_format(col("Date"), "yyyyMMdd").cast("int")) \
    .withColumn("Year", year(col("Date"))) \
    .withColumn("Quarter", quarter(col("Date"))) \
    .withColumn("Month", month(col("Date"))) \
    .withColumn("Day", dayofmonth(col("Date"))) \
    .withColumn("YearMonth", date_format(col("Date"), "yyyyMM").cast("int")) \
    .withColumn("YearQuarter", concat(col("Year"), lit("Q"), col("Quarter"))) \
    .withColumn("MonthName", date_format(col("Date"), "MMMM")) \
    .withColumn("MonthNameShort", date_format(col("Date"), "MMM")) \
    .withColumn("DayName", date_format(col("Date"), "EEEE")) \
    .withColumn("DayNameShort", date_format(col("Date"), "EEE")) \
    .withColumn("DayOfWeek", dayofweek(col("Date"))) \
    .withColumn("DayOfMonth", dayofmonth(col("Date"))) \
    .withColumn("DayOfYear", dayofyear(col("Date"))) \
    .withColumn("WeekOfYear", weekofyear(col("Date"))) \
    .withColumn("ISOWeekOfYear", weekofyear(col("Date"))) \
    .withColumn("CalendarQuarter", concat(lit("Q"), col("Quarter"))) \
    .withColumn("CalendarSemester", when(col("Quarter") <= 2, 1).otherwise(2))

print("Calendar attributes calculated")
dim_date_df.show(5)

In [ ]:
# Calculate derived date attributes
from pyspark.sql import Window as W

# Day of quarter calculation
quarter_window = W.partitionBy("Year", "Quarter").orderBy("Date")
dim_date_df = dim_date_df.withColumn("DayOfQuarter", row_number().over(quarter_window))

# Week of month calculation
dim_date_df = dim_date_df.withColumn(
    "WeekOfMonth",
    floor((dayofmonth(col("Date")) - 1) / 7) + 1
)

# Date flags
dim_date_df = dim_date_df \
    .withColumn("IsWeekend", when(col("DayOfWeek").isin([1, 7]), True).otherwise(False)) \
    .withColumn("IsWeekday", when(col("DayOfWeek").isin([1, 7]), False).otherwise(True)) \
    .withColumn("IsLastDayOfMonth", col("Date") == last_day(col("Date"))) \
    .withColumn("IsLeapYear", 
                when((col("Year") % 4 == 0) & ((col("Year") % 100 != 0) | (col("Year") % 400 == 0)), True)
                .otherwise(False))

# Calculate last day of quarter
dim_date_df = dim_date_df \
    .withColumn("IsLastDayOfQuarter",
                when((col("Month").isin([3, 6, 9, 12])) & (col("Date") == last_day(col("Date"))), True)
                .otherwise(False)) \
    .withColumn("IsLastDayOfYear",
                when((col("Month") == 12) & (col("DayOfMonth") == 31), True)
                .otherwise(False))

print("Derived attributes calculated")

## 4. Fiscal Year Calculations

In [ ]:
# Function to calculate fiscal year attributes
def add_fiscal_year_columns(df, prefix, fiscal_start_month, fiscal_start_day):
    """
    Add fiscal year columns to dataframe
    
    Args:
        df: Input dataframe
        prefix: Column prefix (e.g., 'Group', 'DIV1')
        fiscal_start_month: Fiscal year start month (1-12)
        fiscal_start_day: Fiscal year start day (1-31)
    """
    
    # Calculate fiscal year
    # If current date >= fiscal year start in current year, FY = current year + 1
    # Otherwise FY = current year
    df = df.withColumn(
        f"{prefix}_FiscalYear",
        when(
            (col("Month") > fiscal_start_month) | 
            ((col("Month") == fiscal_start_month) & (col("Day") >= fiscal_start_day)),
            col("Year") + 1
        ).otherwise(col("Year"))
    )
    
    # Create fiscal year start date for the current date
    df = df.withColumn(
        f"{prefix}_FiscalYearStartDate",
        when(
            (col("Month") > fiscal_start_month) | 
            ((col("Month") == fiscal_start_month) & (col("Day") >= fiscal_start_day)),
            to_date(concat(col("Year"), lit(f"-{fiscal_start_month:02d}-{fiscal_start_day:02d}")))
        ).otherwise(
            to_date(concat(col("Year") - 1, lit(f"-{fiscal_start_month:02d}-{fiscal_start_day:02d}")))
        )
    )
    
    # Calculate fiscal day of year
    df = df.withColumn(
        f"{prefix}_FiscalDayOfYear",
        datediff(col("Date"), col(f"{prefix}_FiscalYearStartDate")) + 1
    )
    
    # Calculate fiscal month (1-12 within fiscal year)
    df = df.withColumn(
        f"{prefix}_FiscalMonth",
        floor((col(f"{prefix}_FiscalDayOfYear") - 1) / 30.44) + 1
    ).withColumn(
        f"{prefix}_FiscalMonth",
        when(col(f"{prefix}_FiscalMonth") > 12, 12).otherwise(col(f"{prefix}_FiscalMonth"))
    )
    
    # Calculate fiscal quarter
    df = df.withColumn(
        f"{prefix}_FiscalQuarter",
        floor((col(f"{prefix}_FiscalMonth") - 1) / 3) + 1
    )
    
    # Calculate fiscal week of year
    df = df.withColumn(
        f"{prefix}_FiscalWeekOfYear",
        floor((col(f"{prefix}_FiscalDayOfYear") - 1) / 7) + 1
    )
    
    # Create formatted fiscal period columns
    df = df.withColumn(
        f"{prefix}_FiscalYearQuarter",
        concat(lit("FY"), col(f"{prefix}_FiscalYear"), lit("Q"), col(f"{prefix}_FiscalQuarter"))
    ).withColumn(
        f"{prefix}_FiscalYearMonth",
        concat(lit("FY"), col(f"{prefix}_FiscalYear"), lit("M"), 
               lpad(col(f"{prefix}_FiscalMonth").cast("string"), 2, "0"))
    )
    
    # Drop temporary column
    df = df.drop(f"{prefix}_FiscalYearStartDate")
    
    return df

print("Fiscal year calculation function defined")

In [ ]:
# Add Group fiscal year columns
group_fiscal = config['group_fiscal_year']
dim_date_df = add_fiscal_year_columns(
    dim_date_df,
    "Group",
    group_fiscal['start_month'],
    group_fiscal['start_day']
)

print(f"Group fiscal year calculated (starts {group_fiscal['start_month']}/{group_fiscal['start_day']})")

In [ ]:
# Add fiscal year columns for each division
for division in config['divisions']:
    div_code = division['code']
    fiscal_start_month = division['fiscal_year_start_month']
    fiscal_start_day = division['fiscal_year_start_day']
    
    dim_date_df = add_fiscal_year_columns(
        dim_date_df,
        div_code,
        fiscal_start_month,
        fiscal_start_day
    )
    
    print(f"  {div_code}: Fiscal year calculated (starts {fiscal_start_month}/{fiscal_start_day})")

print(f"\nAll {len(config['divisions'])} division fiscal years calculated")

## 5. Holiday Information

In [ ]:
# Define UK Public Holidays (Bank Holidays)
# Using date-based logic for bank holidays

def calculate_easter(year):
    """
    Calculate Easter Sunday using the Anonymous Gregorian algorithm
    Returns date object for Easter Sunday
    """
    from datetime import date, timedelta
    
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    
    return date(year, month, day)

def get_uk_holidays(year):
    """
    Generate list of UK Public Holidays (Bank Holidays) for a given year
    Returns list of (date, holiday_name) tuples
    """
    from datetime import date, timedelta
    import calendar
    
    holidays = []
    
    # Fixed date holidays
    holidays.append((date(year, 1, 1), "New Year's Day"))
    holidays.append((date(year, 12, 25), "Christmas Day"))
    holidays.append((date(year, 12, 26), "Boxing Day"))
    
    # Easter-based holidays
    easter = calculate_easter(year)
    good_friday = easter - timedelta(days=2)
    easter_monday = easter + timedelta(days=1)
    
    holidays.append((good_friday, "Good Friday"))
    holidays.append((easter_monday, "Easter Monday"))
    
    # Early May Bank Holiday - First Monday in May
    may_cal = calendar.monthcalendar(year, 5)
    mondays = [week[0] for week in may_cal if week[0] != 0]
    if len(mondays) >= 1:
        holidays.append((date(year, 5, mondays[0]), "Early May Bank Holiday"))
    
    # Spring Bank Holiday - Last Monday in May
    may_cal = calendar.monthcalendar(year, 5)
    mondays = [week[0] for week in may_cal if week[0] != 0]
    holidays.append((date(year, 5, mondays[-1]), "Spring Bank Holiday"))
    
    # Summer Bank Holiday - Last Monday in August
    aug_cal = calendar.monthcalendar(year, 8)
    mondays = [week[0] for week in aug_cal if week[0] != 0]
    holidays.append((date(year, 8, mondays[-1]), "Summer Bank Holiday"))
    
    return holidays

# Generate holidays for all years in range
if config['holidays']['enabled']:
    all_holidays = []
    for yr in range(start_year, end_year + 1):
        all_holidays.extend(get_uk_holidays(yr))
    
    # Create holidays dataframe
    holidays_data = [(str(h[0]), h[1]) for h in all_holidays]
    holidays_schema = StructType([
        StructField("Date", StringType(), False),
        StructField("HolidayName", StringType(), False)
    ])
    
    holidays_df = spark.createDataFrame(holidays_data, schema=holidays_schema)
    holidays_df = holidays_df.withColumn("Date", to_date(col("Date")))
    
    # Join holidays with dimension table
    dim_date_df = dim_date_df.join(holidays_df, on="Date", how="left")
    dim_date_df = dim_date_df \
        .withColumn("IsHoliday", col("HolidayName").isNotNull()) \
        .fillna({"HolidayName": ""})
    
    holiday_count = dim_date_df.filter(col("IsHoliday") == True).count()
    print(f"Holidays added: {holiday_count} holiday dates identified")
else:
    dim_date_df = dim_date_df \
        .withColumn("IsHoliday", lit(False)) \
        .withColumn("HolidayName", lit(""))
    print("Holidays disabled in configuration")

## 6. Final Data Preparation

In [ ]:
# Add load timestamp
from pyspark.sql.functions import current_timestamp

dim_date_df = dim_date_df.withColumn("LoadDate", current_timestamp())

# Reorder columns to match table schema
base_columns = [
    "DateKey", "Date",
    "Year", "Quarter", "Month", "Day",
    "YearMonth", "YearQuarter",
    "MonthName", "MonthNameShort",
    "DayName", "DayNameShort",
    "DayOfWeek", "DayOfMonth", "DayOfYear", "DayOfQuarter",
    "WeekOfYear", "WeekOfMonth", "ISOWeekOfYear",
    "CalendarQuarter", "CalendarSemester",
    "IsWeekend", "IsWeekday",
    "IsLastDayOfMonth", "IsLastDayOfQuarter", "IsLastDayOfYear",
    "IsLeapYear",
    "IsHoliday", "HolidayName",
    "Group_FiscalYear", "Group_FiscalQuarter", "Group_FiscalMonth",
    "Group_FiscalYearQuarter", "Group_FiscalYearMonth",
    "Group_FiscalWeekOfYear", "Group_FiscalDayOfYear"
]

# Add division columns
for div in config['divisions']:
    div_code = div['code']
    base_columns.extend([
        f"{div_code}_FiscalYear",
        f"{div_code}_FiscalQuarter",
        f"{div_code}_FiscalMonth",
        f"{div_code}_FiscalYearQuarter",
        f"{div_code}_FiscalYearMonth",
        f"{div_code}_FiscalWeekOfYear",
        f"{div_code}_FiscalDayOfYear"
    ])

base_columns.append("LoadDate")

# Select columns in order
dim_date_final = dim_date_df.select(base_columns)

print(f"Final dataset prepared with {len(base_columns)} columns")
print(f"Total rows: {dim_date_final.count()}")

## 7. Data Quality Checks

In [ ]:
# Quality checks
print("=" * 60)
print("DATA QUALITY CHECKS")
print("=" * 60)

# Check for duplicates
duplicate_count = dim_date_final.groupBy("DateKey").count().filter(col("count") > 1).count()
print(f"\n1. Duplicate DateKeys: {duplicate_count}")
if duplicate_count > 0:
    print("   ⚠️ WARNING: Found duplicate DateKeys!")
else:
    print("   ✓ No duplicates found")

# Check date range
min_date = dim_date_final.agg({"Date": "min"}).collect()[0][0]
max_date = dim_date_final.agg({"Date": "max"}).collect()[0][0]
print(f"\n2. Date Range: {min_date} to {max_date}")

# Check for gaps
expected_days = (datetime.strptime(str(max_date), "%Y-%m-%d") - 
                 datetime.strptime(str(min_date), "%Y-%m-%d")).days + 1
actual_days = dim_date_final.count()
print(f"\n3. Date Continuity:")
print(f"   Expected days: {expected_days}")
print(f"   Actual days: {actual_days}")
if expected_days == actual_days:
    print("   ✓ No gaps in date sequence")
else:
    print(f"   ⚠️ WARNING: Found {expected_days - actual_days} missing dates!")

# Holiday statistics
holiday_count = dim_date_final.filter(col("IsHoliday") == True).count()
print(f"\n4. Holidays: {holiday_count} total holiday dates")

# Weekend statistics
weekend_count = dim_date_final.filter(col("IsWeekend") == True).count()
weekend_pct = (weekend_count / actual_days) * 100
print(f"\n5. Weekends: {weekend_count} days ({weekend_pct:.1f}%)")

# Sample fiscal year data
print(f"\n6. Sample Fiscal Year Data:")
dim_date_final.filter(
    (col("Month") == group_fiscal['start_month']) & 
    (col("Day") == group_fiscal['start_day'])
).select(
    "Date", "Year", "Group_FiscalYear", 
    f"{config['divisions'][0]['code']}_FiscalYear"
).orderBy("Date").show(5)

print("\n" + "=" * 60)
print("Quality checks completed")
print("=" * 60)

## 8. Load Data to Warehouse

In [ ]:
# Load data to warehouse table
print("Loading data to Dim_Date table...")

try:
    # Write to table (overwrite mode)
    dim_date_final.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable("Dim_Date")
    
    print("✓ Data loaded successfully")
    
    # Verify load
    row_count = spark.sql("SELECT COUNT(*) as cnt FROM Dim_Date").collect()[0]['cnt']
    print(f"✓ Verified: {row_count} rows in Dim_Date table")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    raise

In [ ]:
# Display sample of loaded data
print("\nSample data from Dim_Date:")
spark.sql("""
    SELECT 
        DateKey,
        Date,
        Year,
        MonthName,
        DayName,
        IsHoliday,
        HolidayName,
        Group_FiscalYear,
        Group_FiscalQuarter
    FROM Dim_Date
    ORDER BY Date
    LIMIT 10
""").show(10, truncate=False)

## 9. Summary

**Dim_Date Generation Complete!**

The date dimension table has been successfully generated and loaded with:
- Comprehensive calendar attributes
- Group fiscal year calculations
- Multi-division fiscal year support
- UK Public Holiday (Bank Holiday) information
- Data quality validation

**Next Steps:**
1. Run the `Generate_Dim_Time.ipynb` notebook to create the time dimension
2. Schedule this notebook via Data Pipeline for regular refresh
3. Grant appropriate permissions to users/groups